# Week 4 Automate Parametric Simulations

In [ ]:
import pandas as pd
import numpy as np
from eppy.modeleditor import IDF
from eppy import modeleditor
import esoreader
import os

In [ ]:
eplus_root = r"C:\EnergyPlusV25-1-0" # Change this to your EnergyPlus root directory
iddfile = os.path.join(eplus_root,"Energy+.idd")
try:
    IDF.setiddname(iddfile)
except modeleditor.IDDAlreadySetError as e:
    print(e)
    
root_dir = os.getcwd()

MAP_WEATHER = {
    "San Francisco":os.path.normpath(os.path.join(root_dir, r'..\weather_data\USA_CA_San.Francisco.Intl.AP.724940_TMY3.epw')),
    "Sacramento":os.path.normpath(os.path.join(root_dir, r"..\weather_data\USA_CA_Sacramento.Exec.AP.724830_TMY3.epw")),
    "Chicago":os.path.normpath(os.path.join(root_dir, r"..\weather_data\USA_IL_Chicago-OHare.Intl.AP.725300_TMY3.epw")),
    "New York":os.path.normpath(os.path.join(root_dir, r"..\weather_data\USA_NY_New.York-J.F.Kennedy.Intl.AP.744860_TMY3.epw")),
}

In [ ]:
class ESO:
    def __init__(self, path):
        self.dd, self.data = esoreader.read(path)
    def read_var(self, variable, frequency = "Hourly"):
        return [
            {"key": k,
             "series": self.data[self.dd.index[frequency, k, variable]]}
            for _f, k, _v in self.dd.find_variable(variable)
        ]
    def get_df(self, variable, frequency = "Hourly"):
        dic = self.read_var(variable, frequency)
        key = [each["key"] for each in dic]
        values = [each["series"] for each in dic]
        df = pd.DataFrame(values,index = key).T
        return df
    def total_kwh(self, variable, frequency = "Hourly"):
        j_per_kwh = 3_600_000
        results = self.read_var(variable,frequency)
        return sum(sum(s["series"]) for s in results)/j_per_kwh

In [ ]:
# run one single simulation using abs paths
wea_path = r"G:\My Drive\Arch 298\W4_Exercise\weather_data\USA_CA_San.Francisco.Intl.AP.724940_TMY3.epw"

idf_path = r"G:\My Drive\Arch 298\W4_Exercise\Part1\model_1.idf"

idf = IDF(idf_path, wea_path)

output_dir = r"G:\My Drive\Arch 298\W4_Exercise\Part1\temp_output"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

idf.run(output_directory=output_dir)

In [ ]:
# run one single simulation using abs paths
print(root_dir)
if wea == 'San Francisco':
    wea_path = os.path.join(root_dir, r'..\weather_data\USA_CA_San.Francisco.Intl.AP.724940_TMY3.epw')

mod = 1
idf_path = os.path.join(root_dir, f"model_{str(mod)}.idf")

idf = IDF(idf_path, wea_path)

output_dir = r"G:\My Drive\Arch 298\W4_Exercise\Part1\temp_output"

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

idf.run(output_directory=output_dir)

In [ ]:
# read the parametric table 
param_table = pd.read_csv(os.path.join(root_dir,"param_table.csv"), header=0)
param_table.head(3)

# create a result dataframe to store results
result_df = param_table.copy().assign(Heating_kwh=None, Cooling_kwh=None, Lighting_kwh=None)

for index, row in param_table.iterrows():
    
    sim_id = row['sim_id']
    wea = row['weather']
    mod = row['model']
    
    idf_dir = os.path.join(root_dir, f"model_{str(mod)}.idf")
    epw_dir = MAP_WEATHER[wea]
    
    idf = IDF(idf_dir, epw_dir)
    
    # add light electric meter output
    idf.newidfobject("OUTPUT:METER",
                     Key_Name="InteriorLights:Electricity",
                     Reporting_Frequency="TimeStep")
    
    # define sim folder path    
    sim_dir = os.path.join(root_dir, "param_sim", f"sim_{sim_id}")
    
    os.makedirs(sim_dir, exist_ok=True)
    
    idf.saveas(os.path.join(sim_dir, "new_idf.idf"))
        
    try:
        idf.run(expandobjects=True, output_directory=sim_dir)
        
         # read the eso file
        eso_dir = os.path.join(sim_dir,"eplusout.eso")
        eso = ESO(eso_dir)
        
        # get heating, cooling and lighting energy use in kwh
        result_df.loc[index, "Heating_kwh"] = eso.total_kwh("DistrictHeatingWater:Facility","TimeStep")
        result_df.loc[index, "Cooling_kwh"] = eso.total_kwh("DistrictCooling:Facility","TimeStep")
        result_df.loc[index, "Lighting_kwh"] = eso.total_kwh("InteriorLights:Electricity","TimeStep")
            
        print(f'sim {sim_id} done')
    
    except Exception as e:
        print(f"⚠️ Simulation {sim_id} failed: {e}")
    
    os.chdir(root_dir)

# save the result dataframe
result_df.to_csv(os.path.join(root_dir,"part1_results.csv"), index=False)  